In [1]:
import pandas as pd
from mp_api.client import MPRester

/home/salan/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


API_KEY = "iYOf93n7ChLXuvNQq5Mjr7LXsCZYIFct"

elements = ["Zr", "Hf", "Ta", "W", "Mo", "Nb", "Ti", "Cr"]

summary_docs = []

# =========================
# STEP 1 — summary
# =========================
with MPRester(API_KEY) as mpr:
    for el in elements:
        docs = mpr.materials.summary.search(
            elements=[el],
            energy_above_hull=(0, 0.05),
            fields=[
                "material_id",
                "formula_pretty",
                "elements",
                "formation_energy_per_atom",
                "energy_above_hull",
                "density",
                "band_gap",
                "nsites",
                "volume",
                "is_stable"
            ]
        )
        summary_docs.extend(docs)

summary_df = pd.DataFrame([d.dict() for d in summary_docs]).drop_duplicates("material_id")

material_ids = summary_df["material_id"].tolist()

print("Materials found:", len(material_ids))


# =========================
# STEP 2 — elasticity (mechanical strength)
# =========================
elastic_docs = []

with MPRester(API_KEY) as mpr:
    for batch in chunk_list(material_ids, 200):

        docs = mpr.materials.elasticity.search(
            material_ids=batch,
            fields=[
                "material_id",
                "bulk_modulus",
                "shear_modulus",
                "youngs_modulus",
                "universal_anisotropy",
                "homogeneous_poisson",
                "thermal_conductivity",
                "debye_temperature"
            ]
        )

        elastic_docs.extend(docs)

elastic_df = pd.DataFrame([d.dict() for d in elastic_docs])

# =========================
# STEP 3 — thermo extras
# =========================
thermo_docs = []

with MPRester(API_KEY) as mpr:
    for batch in chunk_list(material_ids, 200):

        docs = mpr.materials.thermo.search(
            material_ids=batch,
            fields=[
                "material_id",
                "energy_per_atom",
                "formation_energy_per_atom",
                "energy_above_hull",
                "is_stable"
            ]
        )

        thermo_docs.extend(docs)

thermo_df = pd.DataFrame([d.dict() for d in thermo_docs])



# =========================
# STEP 4 — merge everything
# =========================
df = summary_df.merge(elastic_df, on="material_id", how="left")
df = df.merge(thermo_df, on="material_id", how="left")

df.to_csv("second_material_dataset.csv", index=False)

print("✅ Saved rich dataset:", len(df), "rows")


Retrieving SummaryDoc documents: 100%|██████████| 2696/2696 [00:00<00:00, 3059.95it/s]


Materials found: 16574


Retrieving ElasticityDoc documents: 100%|██████████| 18/18 [00:00<00:00, 70361.11it/s]
Retrieving ElasticityDoc documents: 0it [00:00, ?it/s]
Retrieving ElasticityDoc documents: 100%|██████████| 6/6 [00:00<00:00, 62757.67it/s]
Retrieving ElasticityDoc documents: 0it [00:00, ?it/s]
Retrieving ElasticityDoc documents: 100%|██████████| 10/10 [00:00<00:00, 127875.12it/s]
Retrieving ElasticityDoc documents: 0it [00:00, ?it/s]
Retrieving ElasticityDoc documents: 100%|██████████| 5/5 [00:00<00:00, 47127.01it/s]
Retrieving ElasticityDoc documents: 0it [00:00, ?it/s]
Retrieving ElasticityDoc documents: 0it [00:00, ?it/s]
Retrieving ElasticityDoc documents: 100%|██████████| 1/1 [00:00<00:00, 6364.65it/s]
Retrieving ElasticityDoc documents: 0it [00:00, ?it/s]
Retrieving ThermoDoc documents: 100%|██████████| 384/384 [00:00<00:00, 4364804.16it/s]


✅ Saved rich dataset: 38646 rows


In [7]:
df = pd.read_csv("second_material_dataset.csv")

In [8]:
df.sample(10)

,nsites,elements,formula_pretty,volume,density,material_id,formation_energy_per_atom_x,energy_above_hull_x,is_stable_x,band_gap,...,thermal_conductivity,universal_anisotropy,homogeneous_poisson,debye_temperature,fields_not_requested_y,energy_per_atom,formation_energy_per_atom_y,energy_above_hull_y,is_stable_y,fields_not_requested
3566,8,"['Sc', 'Zr']",Zr3Sc,188.776898,5.605496,mp-cppms,-0.015424,0.000000,True,0.0000,...,NaN,NaN,NaN,NaN,NaN,-19.927390,-0.015424,0.000000,True,"['builder_meta', 'nsites', 'elements', 'neleme..."
3644,32,"['Ge', 'Zr']",Zr3Ge,675.258232,6.812974,mp-cqmbs,-0.605558,0.000000,True,0.0000,...,NaN,NaN,NaN,NaN,NaN,-21.135896,-0.605558,0.000000,True,"['builder_meta', 'nsites', 'elements', 'neleme..."
25126,40,"['Ba', 'Li', 'Nb', 'O']",Ba4LiNb3O12,582.342157,5.856724,mp-bbaw,-3.206630,0.000000,True,2.5798,...,NaN,NaN,NaN,NaN,NaN,-8.169844,-3.206630,0.000000,True,"['builder_meta', 'nsites', 'elements', 'neleme..."
19308,40,"['Ba', 'Mo', 'O', 'P']",BaMo2P3O14,541.909261,3.959730,mp-bfphr,-2.573750,0.000000,True,1.8697,...,NaN,NaN,NaN,NaN,NaN,-11.940908,-2.477646,0.000000,True,"['builder_meta', 'nsites', 'elements', 'neleme..."
38329,56,"['Cr', 'Na', 'O', 'P']",NaCr2P2O9,687.381931,3.217038,mp-chfds,-2.519497,0.001852,False,0.7364,...,NaN,NaN,NaN,NaN,NaN,-7.948812,-2.521178,0.001852,False,"['builder_meta', 'nsites', 'elements', 'neleme..."
35544,14,"['Cr', 'In', 'Mg', 'S']",Mg2CrIn3S8,308.182475,3.780232,mp-csyus,-1.094245,0.031131,False,1.1668,...,NaN,NaN,NaN,NaN,NaN,-4.972151,-1.100704,0.024084,False,"['builder_meta', 'nsites', 'elements', 'neleme..."
8487,4,"['As', 'Ta']",TaAs,69.585364,12.211800,mp-cwm,-0.620701,0.000000,True,0.0000,...,"{'clarke': 0.7840991806991481, 'cahill': 0.864...",0.666,0.276,352.842098,"['builder_meta', 'nsites', 'elements', 'neleme...",-8.801826,-0.543378,0.000000,True,"['builder_meta', 'nsites', 'elements', 'neleme..."
22416,20,"['Na', 'Nb', 'O']",NaNbO3,260.553398,4.178076,mp-bfvlq,-2.805655,0.039168,False,2.0583,...,NaN,NaN,NaN,NaN,NaN,-8.063037,-2.805655,0.039168,False,"['builder_meta', 'nsites', 'elements', 'neleme..."
2169,40,"['O', 'Sr', 'V', 'Zr']",Sr2ZrVO6,532.120457,5.160260,mp-cuqpb,-3.272632,0.017182,False,1.1541,...,NaN,NaN,NaN,NaN,NaN,-8.343055,-3.272632,0.017182,False,"['builder_meta', 'nsites', 'elements', 'neleme..."
38423,58,"['B', 'Cr', 'Na', 'O', 'S']",Na6Cr2B4SO16,692.427702,2.749373,mp-bwkpf,-2.373896,0.049643,False,0.0000,...,NaN,NaN,NaN,NaN,NaN,-7.111094,-2.373896,0.049643,False,"['builder_meta', 'nsites', 'elements', 'neleme..."
